## Setup

Importings libs and setting up data frames and some helper functions...

In [38]:
# Import necessary libraries
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from enum import IntFlag

plt.style.use('ggplot')
%matplotlib inline

In [39]:
path = r"C:\Users\Kuinox\Documents\parquet_output"
addresses_file = os.path.join(path, 'addresses.parquet')
symbols_file = os.path.join(path, 'symbols.parquet')
tracesamples_file = os.path.join(path, 'tracesamples.parquet')
comms_file = os.path.join(path, 'comms.parquet')
aux_events_file = os.path.join(path, 'aux_events.parquet')

In [40]:
addresses_df = pl.scan_parquet(addresses_file).lazy()
symbols_df = pl.scan_parquet(symbols_file).lazy()
tracesamples_df = pl.scan_parquet(tracesamples_file).lazy()
comms_df = pl.scan_parquet(comms_file).lazy()
aux_events_df = pl.scan_parquet(aux_events_file).lazy()


## Helpers methods

In [41]:

def df_to_dict(df):
    collected = df.collect()
    return dict(zip(collected['id'], collected['string']))



def map_column_values(df, old_col_name, new_col_name, mapping_dict):
    return df.with_columns(
        pl.col(old_col_name).map_elements(
            lambda x: mapping_dict.get(x, "Unknown"), 
            return_dtype=pl.Utf8
        ).alias(new_col_name)
    )

comms_id2str = df_to_dict(comms_df)
sym_id2str = df_to_dict(symbols_df)
addresses_df = map_column_values(addresses_df, 'commStrId', 'command_name', comms_id2str)
addresses_df = map_column_values(addresses_df, 'symStrId', 'symbol_name', sym_id2str)

## How the raw data looks like...

In [42]:
print("--- Addresses Schema ---")
display(addresses_df.collect_schema())

print("\n--- Symbols Schema ---")
display(symbols_df.collect_schema())

print("\n--- Trace Samples Schema ---")
display(tracesamples_df.collect_schema())

print("\n--- Aux Events Schema ---")
display(aux_events_df.collect_schema())


addresses_count = addresses_df.select(pl.len()).collect().item()
symbols_count = symbols_df.select(pl.len()).collect().item()
tracesamples_count = tracesamples_df.select(pl.len()).collect().item()
aux_events_count = aux_events_df.select(pl.len()).collect().item()

print(f"Addresses count: {addresses_count:,}")
print(f"Symbols count: {symbols_count:,}")
print(f"Trace samples count: {tracesamples_count:,}")
print(f"Comms count: {comms_df.select(pl.len()).collect().item():,}")
print(f"Aux events count: {aux_events_count:,}")

threads_summary_df = (tracesamples_df
            .join(addresses_df, left_on='id', right_on='traceId', how='inner')
            .select(['tid', 'commStrId'])
            .filter(pl.col('commStrId') != 0)
            .unique()
            .group_by('tid')
            .agg(
                pl.col('commStrId').map_elements(
                    lambda id_val: comms_id2str.get(id_val.item() if isinstance(id_val, pl.Series) and id_val.len() == 1 else id_val, "Unnamed"),
                    return_dtype=pl.Utf8
                ).implode().alias('command_names_list')
            )
            .with_columns(
                pl.col('command_names_list').list.join("/").alias('command_names_str')
            )
            .select(['tid', 'command_names_str'])
            .collect()
)

tid_to_command_str_map = dict(zip(threads_summary_df['tid'], threads_summary_df['command_names_str']))
print("\n--- Threads ---")
print(tid_to_command_str_map)



--- Addresses Schema ---


Schema([('id', UInt64),
        ('traceId', UInt64),
        ('address', UInt64),
        ('pid', UInt32),
        ('isIp', Boolean),
        ('size', UInt32),
        ('symoff', UInt32),
        ('symStrId', UInt64),
        ('symStart', UInt64),
        ('symEnd', UInt64),
        ('dso', UInt64),
        ('symBinding', UInt8),
        ('is64Bit', UInt8),
        ('isKernelIp', UInt8),
        ('buildId', Binary),
        ('filtered', UInt8),
        ('commStrId', UInt64),
        ('priv', UInt64),
        ('command_name', String),
        ('symbol_name', String)])


--- Symbols Schema ---


Schema([('id', UInt64), ('string', String)])


--- Trace Samples Schema ---


Schema([('id', UInt64),
        ('perfId', UInt64),
        ('pid', UInt32),
        ('tid', UInt32),
        ('time', UInt64),
        ('flags', UInt32),
        ('cpu', UInt32),
        ('ip', UInt64),
        ('addr', UInt64),
        ('period', UInt64),
        ('insnCnt', UInt64),
        ('cycCnt', UInt64),
        ('weight', UInt64),
        ('cpumode', UInt8),
        ('addrCorrelatesSym', UInt8),
        ('event', String),
        ('machinePid', UInt32),
        ('vcpu', UInt32)])


--- Aux Events Schema ---


Schema([('time', UInt64),
        ('pid', UInt64),
        ('tid', UInt64),
        ('cpu', UInt64),
        ('flags', UInt64)])

Addresses count: 232,917,414
Symbols count: 720
Trace samples count: 212,496,770
Comms count: 8
Aux events count: 897
{18463: '.NET SynchManag', 18468: '.NET Tiered Com', 18464: '.NET EventPipe', 18469: '.NET SigHandler', 18466: '.NET Debugger', 18467: '.NET Finalizer', 18461: 'dotnet', 18465: '.NET DebugPipe'}


## Pre Processing 
The purpose of the tool we used before are just to to quickly collect and pack the data in a format accessible.  
Also, it allow you to work on this even if you don't run linux, bare metal, with an intel cpu.

### Checking assumptions

In [43]:
# are all instruction ordered by time ?
time_order_check = tracesamples_df.select(
    pl.col('time').diff().lt(0).sum().alias('out_of_order_count')
).collect()

out_of_order_count = time_order_check[0, 'out_of_order_count']

if out_of_order_count == 0:
    print("All instructions are correctly ordered by time.")
else:
    print(f"Found {out_of_order_count} instances where instructions are not ordered by time.")
    raise ValueError("Time sequence error detected in trace samples. Processing halted.")

All instructions are correctly ordered by time.


### Aux Events

While the other events happen from the DLFilter, this one come from directly parsing the file.  
Why ? `perf` need to run on the target machine and allow us to fetch easily symbols infos from an instruction. But it only pass samples to the dlfilter.  
There is an important informations in the events: AUX data loss infos.  
This tell us when intel_pt buffer got saturated and it had to drop events.

In [44]:
aux_events_df = aux_events_df.with_columns(
    (pl.col('flags') != 0).alias('is_error')
)

aux_data_loss = aux_events_df.filter(pl.col('is_error'))
aux_data_loss_count = aux_data_loss.select(pl.len()).collect().item()
print(f"Aux data loss count: {aux_data_loss_count}")

Aux data loss count: 13


### Samples

In [ ]:
class SampleFlags(IntFlag):
    BRANCH       = 1 << 0
    CALL         = 1 << 1
    RETURN       = 1 << 2
    CONDITIONAL  = 1 << 3
    SYSCALLRET   = 1 << 4
    ASYNC        = 1 << 5
    INTERRUPT    = 1 << 6
    TX_ABORT     = 1 << 7
    TRACE_BEGIN  = 1 << 8
    TRACE_END    = 1 << 9
    IN_TX        = 1 << 10
    VMENTRY      = 1 << 11
    VMEXIT       = 1 << 12

tracesamples_df = tracesamples_df.with_columns([
        (pl.col('flags') & SampleFlags.BRANCH > 0).alias('is_branch'),
        (pl.col('flags') & SampleFlags.CALL > 0).alias('is_call'),
        (pl.col('flags') & SampleFlags.RETURN > 0).alias('is_return'),
        (pl.col('flags') & SampleFlags.CONDITIONAL > 0).alias('is_conditional'),
        (pl.col('flags') & SampleFlags.SYSCALLRET > 0).alias('is_syscallret'),
        (pl.col('flags') & SampleFlags.ASYNC > 0).alias('is_async'),
        (pl.col('flags') & SampleFlags.INTERRUPT > 0).alias('is_interrupt'),
        (pl.col('flags') & SampleFlags.TX_ABORT > 0).alias('is_tx_abort'),
        (pl.col('flags') & SampleFlags.TRACE_BEGIN > 0).alias('is_trace_begin'),
        (pl.col('flags') & SampleFlags.TRACE_END > 0).alias('is_trace_end'),
        (pl.col('flags') & SampleFlags.IN_TX > 0).alias('is_in_tx'),
        (pl.col('flags') & SampleFlags.VMENTRY > 0).alias('is_vmentry'),
        (pl.col('flags') & SampleFlags.VMEXIT > 0).alias('is_vmexit')
])

tracesamples_df = tracesamples_df.with_columns([
    pl.col("tid").alias("threadId"),
    pl.col("pid").alias("processId"),
    pl.col("insnCnt").alias("instructionCount"),
    pl.col("cycCnt").alias("cycleCount"),
    pl.col("addrCorrelatesSym").alias("addressCorrelatesSymbol")
])

flag_infos = (
    tracesamples_df
    .select([
        pl.sum('is_branch').alias('branch_count'),
        pl.sum('is_call').alias('call_count'),
        pl.sum('is_return').alias('return_count'),
        pl.sum('is_conditional').alias('conditional_count'),
        pl.sum('is_syscallret').alias('syscallret_count'),
        pl.sum('is_async').alias('async_count'),
        pl.sum('is_interrupt').alias('interrupt_count'),
        pl.sum('is_tx_abort').alias('tx_abort_count'),
        pl.sum('is_trace_begin').alias('trace_begin_count'),
        pl.sum('is_trace_end').alias('trace_end_count'),
        pl.sum('is_in_tx').alias('in_tx_count'),
        pl.sum('is_vmentry').alias('vmentry_count'),
        pl.sum('is_vmexit').alias('vmexit_count')
    ])
    .collect()
)

instruction_counts = tracesamples_df.select(pl.len()).collect().item()
print(f"Instruction counts: {instruction_counts}")

print("Branch type counts:")
display(flag_infos)

unique_thread_ids = tracesamples_df.select('threadId').unique().collect()['threadId'].to_list()
dfs_by_thread = {
    tid: tracesamples_df.filter(pl.col('threadId') == tid).lazy()
    for tid in unique_thread_ids
}
print(f"Type of dfs_by_thread: {type(dfs_by_thread)}")


print(f"\nCreated {len(dfs_by_thread)} DataFrames, one per thread.")

split_samples_by_thread = {}

print("\n--- Starting processing of all threads ---")
for outer_loop_count, (thread_id, thread_df_lazy) in enumerate(dfs_by_thread.items()):
    thread_name = tid_to_command_str_map.get(thread_id, f"Unknown Thread: {thread_id}")
    print(f"\n[Outer Loop Iteration: {outer_loop_count}, Thread ID: {thread_id}] Processing splits for thread: {thread_name}")
    
    # Filter aux_data_loss for the current thread
    thread_specific_aux_data_loss = aux_data_loss.filter(pl.col('tid') == thread_id)
    thread_specific_loss_times_df = thread_specific_aux_data_loss.select(pl.col('time')).collect()
    
    current_thread_splits = []
    prev_time = 0
    
    loss_times_list = thread_specific_loss_times_df['time'].to_list()
    num_loss_events = len(loss_times_list)
    print(f"  [Thread ID: {thread_id}] Found {num_loss_events} AUX data loss event(s) for thread '{thread_name}'.")

    if num_loss_events == 0:
        print(f"    [Thread ID: {thread_id}] num_loss_events is 0. Treating as a single segment.")
        current_thread_splits.append(thread_df_lazy)
    else:
        print(f"    [Thread ID: {thread_id}] num_loss_events is {num_loss_events}. Splitting into {num_loss_events + 1} segments.")
        for time_val in loss_times_list:
            segment = thread_df_lazy.filter((pl.col('time') > prev_time) & (pl.col('time') <= time_val))
            current_thread_splits.append(segment)
            prev_time = time_val
        
        # Add the remaining samples after the last aux data loss for this thread
        remaining_segment = thread_df_lazy.filter(pl.col('time') > prev_time)
        current_thread_splits.append(remaining_segment)
    
    split_samples_by_thread[thread_id] = current_thread_splits
    
    print(f"  [Thread ID: {thread_id}] Before inner loop, current_thread_splits has {len(current_thread_splits)} element(s).")
    for i, sample_df_lazy in enumerate(current_thread_splits):
        # Collect here to get the shape for printing
        collected_sample_df = sample_df_lazy.collect()
        print(f"    [Thread ID: {thread_id}, Inner Loop Segment Index: {i}] Segment {i + 1}: {collected_sample_df.shape[0]:,} instructions")

print("\n--- Finished processing all threads ---")

Instruction counts: 212496770
Branch type counts:


branch_count,call_count,return_count,conditional_count,syscallret_count,async_count,interrupt_count,tx_abort_count,trace_begin_count,trace_end_count,in_tx_count,vmentry_count,vmexit_count
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
821738,212470735,136304216,0,0,0,0,0,0,0,0,0,0


Type of dfs_by_thread: <class 'dict'>

Created 8 DataFrames, one per thread.
Thread ID: 18466
Thread ID: 18463
Thread ID: 18469
Thread ID: 18467
Thread ID: 18461
Thread ID: 18464
Thread ID: 18468
Thread ID: 18465
Thread ID: 18466

Processing splits for thread: .NET Debugger (ID: 18466)
  Found 0 aux data loss events for thread .NET Debugger.
  Segment 1: 10,201 instructions
Thread ID: 18463

Processing splits for thread: .NET SynchManag (ID: 18463)
  Found 0 aux data loss events for thread .NET SynchManag.
  Segment 1: 4,571 instructions
Thread ID: 18469

Processing splits for thread: .NET SigHandler (ID: 18469)
  Found 0 aux data loss events for thread .NET SigHandler.
  Segment 1: 1,708 instructions
Thread ID: 18467

Processing splits for thread: .NET Finalizer (ID: 18467)
  Found 1 aux data loss events for thread .NET Finalizer.
  Segment 1: 4,571 instructions
Thread ID: 18469

Processing splits for thread: .NET SigHandler (ID: 18469)
  Found 0 aux data loss events for thread .NET S

### Stack Reconstruction

Now we can reconstruct the stack for each instruction series.

### 

# Overview

In [46]:
threads_info = (tracesamples_df
            .join(addresses_df, left_on='id', right_on='traceId', how='inner')
            .select(['tid', 'commStrId'])
            .unique()
            .group_by('tid')
            .agg(
                pl.count('commStrId').alias('command_count'),
                pl.col('commStrId').implode().alias('command_ids')
            )
            .collect()
)


threads_info = threads_info.with_columns(
    pl.col('command_ids').map_elements(
        lambda ids: [get_command_name(cmd_id) for cmd_id in ids],
        return_dtype=pl.List(pl.Utf8)
    ).alias('command_names')
)

print(threads_info)


shape: (8, 4)
┌───────┬───────────────┬─────────────┬────────────────────────────────┐
│ tid   ┆ command_count ┆ command_ids ┆ command_names                  │
│ ---   ┆ ---           ┆ ---         ┆ ---                            │
│ u32   ┆ u32           ┆ list[u64]   ┆ list[str]                      │
╞═══════╪═══════════════╪═════════════╪════════════════════════════════╡
│ 18461 ┆ 2             ┆ [1, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18464 ┆ 2             ┆ [0, 3]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18466 ┆ 2             ┆ [5, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18467 ┆ 2             ┆ [6, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18465 ┆ 2             ┆ [4, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18469 ┆ 2             ┆ [8, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18463 ┆ 2             ┆ [2, 0]      ┆ ["Unknown TID", "Unknown TID"] │
│ 18468 ┆ 2             ┆ [7, 0]      ┆ ["Unknown TID", "Unknown TID"] │
└───────┴───────────────┴────────────